# 롤 API 데이터 수집하기

In [1]:
import requests
import json
import time
import numpy as np
import pandas as pd

## API를 통한 소환사 정보 수집

In [92]:
# Lol Api에서 소환사 명을 통해서 게임 정보를 알 수 있는 puuid 불러오기 
summoner_Name = 'one summer drive' #소환사명
api_key = "RGAPI-3973a848-ffe1-4fe4-a503-827c3a5eef42" # api key
request_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/103.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Charset": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://developer.riotgames.com",
    "X-Riot-Token": "RGAPI-3973a848-ffe1-4fe4-a503-827c3a5eef42"
}

summoner_url="https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key

In [129]:
# requests를 통해서 puuid 추출하기
def summoner(summoner_Name, api_key):
    summoner_url = "https://kr.api.riotgames.com/lol/summoner/v4/summoners/by-name/"+summoner_Name+"/""?api_key="+api_key
    return summoner_r.json()['puuid']

## 수집한 소환사 정보(puuid)를 통해서 매치, 게임정보 확인하기

In [102]:
# puuid를 통한 match를 확인할 수 있는 url 
match_url = "https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/"+summoner_puuid+"/ids?start=0&count=10"
match_r = requests.get(match_url, headers=request_headers)

# 가장 최근에 트롤을 만났던 게임 id 불러오기 
# 트롤은 워윅
match_id = match_r.json()[1]

# 게임에 대한 정보 불러오기
game_url = "https://asia.api.riotgames.com/lol/match/v5/matches/"+match_id
game_r = requests.get(game_url, headers=request_headers)

# 게임 세부내용 저장하기 
game_content = game_r.json()['info']['participants']

## 분석에 사용할 데이터프레임 생성하기

In [110]:
game_content[0]['timePlayed']

1171

In [116]:
int(round(game_content[0]['timePlayed']/60,2)*100)

1952

In [117]:
# 게임 데이터를 담을 DataFrame 생성하기
game_df = pd.DataFrame(columns=['puuid', 'teamPosition', 'individualPosition', 'champName', 'champExp', 'summoner_spell1', 'summoner_spell2' ,
                                'kills', 'deaths', 'assists', 'damageDealtToBuildings', 'totalDamageDealtToChamp', 'damagePerMin', 'teamDamagePer(%)',
                                'killParticipation(%)', 'totalDamageTaken', 'damageTakeOnTeamPer(%)', 'goldEarned', 'goldPerMin', 'totalCs',
                                'maxCsAdvantageOnLaneOpponent', 'maxLevelLeadLaneOpponent', 'gameEndedInEarlySurren', 'gameEndedInSurren',
                                'teamEarlySurrn', 'timeplayed'])

In [118]:
for i in range(len(game_content)):
    input_data = {
        'puuid':game_content[i]['puuid'], 
        'teamPosition':game_content[i]['teamPosition'], 
        'individualPosition':game_content[i]['individualPosition'], 
        'champName':game_content[i]['championName'], 
        'champExp':game_content[i]['champExperience'], 
        'summoner_spell1':game_content[i]['summoner1Id'], 
        'summoner_spell2':game_content[i]['summoner2Id'],
        'kills':game_content[i]['kills'], 
        'deaths':game_content[i]['deaths'], 
        'assists':game_content[i]['assists'], 
        'damageDealtToBuildings':game_content[i]['damageDealtToBuildings'], 
        'totalDamageDealtToChamp':game_content[i]['totalDamageDealtToChampions'],
        'damagePerMin':game_content[i]['challenges']['damagePerMinute'], 
        'teamDamagePer(%)':round(game_content[i]['challenges']['teamDamagePercentage'], 2)*100,
        'killParticipation(%)':game_content[i]['challenges']['killParticipation']*100,
        'totalDamageTaken':game_content[i]['totalDamageTaken'],
        'damageTakeOnTeamPer(%)':round(game_content[i]['challenges']['damageTakenOnTeamPercentage'],2)*100,
        'goldEarned':game_content[i]['goldEarned'],
        'goldPerMin':game_content[i]['challenges']['goldPerMinute'],
        'totalCs':game_content[i]['totalMinionsKilled']+game_content[i]['neutralMinionsKilled'],
        'maxCsAdvantageOnLaneOpponent':game_content[i]['challenges']['maxCsAdvantageOnLaneOpponent'],
        'maxLevelLeadLaneOpponent':game_content[i]['challenges']['maxLevelLeadLaneOpponent'],
        'gameEndedInEarlySurren':game_content[i]['gameEndedInEarlySurrender'],
        'gameEndedInSurren':game_content[i]['gameEndedInSurrender'],
        'teamEarlySurrn':game_content[i]['teamEarlySurrendered'],
        'timePlayed':int(round(game_content[i]['timePlayed']/60,2)*100) # 4자리로 앞의 2자리는 분, 뒤의 2자리는 초로 구성되어있다. 
    }

    game_df = game_df.append(input_data, ignore_index=True)


C:\Users\lima\AppData\Local\Temp\ipykernel_17448\2122601624.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  game_df = game_df.append(input_data, ignore_index=True)
C:\Users\lima\AppData\Local\Temp\ipykernel_17448\2122601624.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  game_df = game_df.append(input_data, ignore_index=True)
C:\Users\lima\AppData\Local\Temp\ipykernel_17448\2122601624.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  game_df = game_df.append(input_data, ignore_index=True)
C:\Users\lima\AppData\Local\Temp\ipykernel_17448\2122601624.py:31: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  game_df = game_df.append

In [119]:
game_df

,puuid,teamPosition,individualPosition,champName,champExp,summoner_spell1,summoner_spell2,kills,deaths,assists,...,goldEarned,goldPerMin,totalCs,maxCsAdvantageOnLaneOpponent,maxLevelLeadLaneOpponent,gameEndedInEarlySurren,gameEndedInSurren,teamEarlySurrn,timeplayed,timePlayed
0,nxp0FYb8Bm2X7Is7P_8t_LytQBIIU-48MqPrBgruO4dVeX...,TOP,TOP,Kayle,10480,12,4,2,1,0,...,7990,409.279655,154,36,1,False,True,False,NaN,1952.0
1,nNNLQ9ep-05WYcCktuee1yPkKoJArsZfjz-jewYiXP8310...,JUNGLE,JUNGLE,XinZhao,8740,4,11,4,0,5,...,8014,410.500117,109,17.75,2,False,True,False,NaN,1952.0
2,vyNBPW8FqdzdM5mRglOnlWc_rNOpsJdWozP_XFGNkrkusk...,MIDDLE,MIDDLE,Ahri,9627,14,4,4,1,3,...,7960,407.711944,131,31,2,False,True,False,NaN,1952.0
3,fBc7j9IjOXOupOnN4UFWcNZ0yD9ZAVcUjsooQU8-cBP_JK...,BOTTOM,BOTTOM,Kaisa,8049,7,4,5,1,7,...,9036,462.822230,138,8,2,False,True,False,NaN,1952.0
4,utdh9SrrlLE4o-xkJbNVuyp2CRRW5kwkJuKEanF0KfL5L_...,UTILITY,UTILITY,Sylas,7011,14,4,5,2,5,...,7211,369.370757,26,7,2,False,True,False,NaN,1952.0
5,cQ5NsmQunQmf0Lo8WMNEx7PZpGwr5g6sbqBmYt2hYIt-72...,TOP,TOP,Gangplank,8837,4,12,1,2,0,...,5923,303.391759,118,7,1,False,True,False,NaN,1952.0
6,sa7tciOemv-8u9L4U4QkKFAQr4usIm1g-kpPz2taJs0kpL...,JUNGLE,JUNGLE,Warwick,6319,4,11,1,4,1,...,5517,282.597147,92,11.0,1,False,True,False,NaN,1952.0
7,ouwnU00p3VVAHZTOQE5ATOb5CWLX0cNh2Uty74UejyKlHX...,MIDDLE,MIDDLE,Xerath,7424,1,4,0,4,0,...,5037,257.990071,101,0,0,False,True,False,NaN,1952.0
8,yo7QnFk5PU5C_1HrguUgJmhyOUS1EDwUwNjBZXGVJw2vAB...,BOTTOM,BOTTOM,Ashe,6626,7,4,3,4,1,...,6636,339.910420,137,15,1,False,True,False,NaN,1952.0
9,rnBD6bHESO4YDkYUvCoqrTzLSSJPBayfyT-eDUv-WuXfxL...,UTILITY,UTILITY,Lux,5286,14,4,0,6,4,...,4439,227.370945,19,4,1,False,True,False,NaN,1952.0


## 가설 확인하기

```
인게임 속의 트롤링은 처음부터 발생할 수도 있고 중간에 소환사의 변심으로 인해서 발생할 수도 있고 수 많은 경우로 인해 발생한다.
기본적으로 의도적인 죽음, 다른 소환사들 방해, 한타에 합류를 하지 않거나, 우물에서 잠수를 하거나 
위와 같은 행위들은 기본적으로 DPM의 하락을 가져온다. 또한 안정적으로 성장을 한 상대 라이너에 비해서 성장 측면에서 많은 차이가 나타난다. 

픽창에서의 싸움으로 인한 소환사 스펠, 주 포지션이 아니므로 인한 트롤 등에 대해서는 소환사 스펠을 통해서 확인한다. 
```

- 가장 최근 게임의 트롤은 워익이었다. 
- 욕설과 함께 한타참여X, 정글링만 하거나, 우물에서 잠수 
- 딜량에 대해서 수치화 하기 
- 게임은 빠른 서렌이 나오는 15분부터 끝나기 시작한다.
- 15분까지는 기본적으로 라인전이 끝나는 시기, 탑, 미드, 바텀의 경우에는 라인전을 통해서 딜량을 쌓을수 있으나 정글의 경우에는 힘들다. 
- 게임이 장기간으로 가는경우 서포터는 딜량이 작기 마련이다. 
- 원딜의 경우에는 코어가 뜨면 뜰 수록 강해진다. 
- 위와 같은 경우를 수치화 조정해서 딜량을 계산한다.

In [ ]:
# timePlayed를 통해서 딜량을 수치 조정한다. 
# 플레이한 20판을 통해서 평균 수치를 확인한다.

In [127]:
# puuid를 통한 match를 확인할 수 있는 url 
match_url = "https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/"+summoner_puuid+"/ids?start=0&count=20"
match_r = requests.get(match_url, headers=request_headers)

In [128]:
len(match_r.json())

20

In [ ]:
for i in len(math_r.json()):
    game_url